In [ ]:
import os
import pandas as pd

In [ ]:
#step 1 and 2 genomic files here
data='/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results'

#pheno and covar file here
results='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results'

In [ ]:
#setup regenie

In [ ]:
!conda config --prepend pkgs_dirs $HOME/.conda/pkgs
!conda config --prepend envs_dirs $HOME/.conda/envs

!conda config --show pkgs_dirs
!conda config --show envs_dirs

In [ ]:
%%bash
# If you already have conda, just do the 'create' line; otherwise see the conda note below.
conda create -n regenie4 -y -c conda-forge -c bioconda regenie

regenie --version  
# should print 4.1.x
which -a regenie           # ensure the conda one is first on PATH


In [ ]:
!conda run -n regenie4 regenie --version
!conda run -n regenie4 which -a regenie

In [ ]:
#Fix genomic files

In [ ]:
#fixstep 1 psam file

def fix_psam_for_regenie(psam_path):
    
   
    # Read whitespace-delimited .psam, keep everything as string
    df = pd.read_csv(psam_path, delim_whitespace=True, dtype=str)

    # Case 1: already has '#FID' and 'IID' (your current case)
    if "#FID" in df.columns and "IID" in df.columns:
        pass

    # Case 2: older file where you only had '#IID'
    elif "#IID" in df.columns and "#FID" not in df.columns:
        # create FID = IID and rename '#IID' -> 'IID'
        df.insert(0, "#FID", df["#IID"])
        df.rename(columns={"#IID": "IID"}, inplace=True)

    else:
        raise ValueError(f"PSAM header columns unexpected: {df.columns.tolist()}")

    # Ensure SEX column exists
    if "SEX" not in df.columns:
        df["SEX"] = "0"  # unknown sex
    else:
        # Fill NaNs and empty strings with "0"
        df["SEX"] = df["SEX"].fillna("0")
        df["SEX"] = df["SEX"].replace("", "0")

    # Write back as tab-delimited, keeping '#FID' as the first column name
    df.to_csv(psam_path, sep="\t", index=False)

    print("Fixed PSAM written to:", psam_path)
    print(df.head())

fix_psam_for_regenie("/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_threshold.step1_snps.psam")

In [ ]:
%%bash

PFX="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_threshold.step1_snps"

plink2 \
  --pfile "${PFX}" \
  --export vcf-4.2 bgz \
  --out tmp_acaf_step1

In [ ]:
!bcftools norm -m-any --threads 16 tmp_acaf_step1.vcf.gz -Oz -o tmp_acaf_step1_split2.vcf.gz
!bcftools index --threads 16 tmp_acaf_step1_split2.vcf.gz

In [ ]:
%%bash

plink2 \
  --vcf tmp_acaf_step1_split2.vcf.gz \
  --threads 16 \
  --make-bed \
  --set-all-var-ids '@:#:$r:$a' \
  --rm-dup force-first \
  --out acaf_step1_regenie
  



In [ ]:
%%bash

plink2 \
  --bfile acaf_step1_regenie \
  --threads 16 \
  --set-all-var-ids '@:#:$r:$a' \
  --new-id-max-allele-len 1000 \
  --make-just-bim \
  --out acaf_step1_regenie_fixed
  

mv acaf_step1_regenie_fixed.bim acaf_step1_regenie.bim

In [ ]:
%%bash

plink2 \
  --bfile acaf_step1_regenie \
  --threads 16 \
  --maf .05 \
  --write-snplist \
  --out regenie_step1_maf5

In [ ]:
!awk 'NR==FNR { chr[$2]=$1; next } ($1 in chr) { print chr[$1] }' \
  acaf_step1_regenie.bim regenie_step1_maf5.snplist \
| sort | uniq -c

In [ ]:
!df -h

In [ ]:
!ps aux | grep plink2

In [ ]:
#run regenie

In [ ]:
%%writefile regenie_test.sh

#!/usr/bin/env bash
set -euo pipefail

# Use REGENIE from your conda env (no need to activate)
export PATH="$HOME/.conda/envs/regenie4/bin:$PATH"


# Show which regenie we will use (sanity check)
echo "Using regenie at: $(which regenie)"


# Local paths

PLINK1="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_step1_regenie"     # prefix only (has .bed/.bim/.fam)

PLINK2="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_v7_plink_files/acaf_threshold.chr16.bed"       # prefix only (has .bed/.bim/.fam)


MERGED_TSV='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt'        # columns: FID IID meningitis sex age

MERGED_TSV2='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt'        # columns: FID IID meningitis sex age

OUT_DIR='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results'

mkdir -p "$OUT_DIR"


#STEP 1 - building moc

regenie \
  --step 1 \
  --bed "${PLINK1}" \
  --extract "regenie_step1_maf5.snplist" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" \
  --phenoCol case \
  --covarFile "${MERGED_TSV}" \
  --covarColList sex,age,age2,age_sex,age2_sex,PC{1:16} \
  --lowmem \
  --bsize 1000 \
  --threads 16 \
  --gz \
  --print-pheno \
  --out "${OUT_DIR}/lupus_all_fit_step1"

# STEP 2 — single-variant test
regenie \
  --step 2 \
  --bed "${PLINK2}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" 
  --phenoCol case \
  --covarFile  "${MERGED_TSV}" \
  --covarColList sex,age,age2,age_sex,age2_sex,PC{1:16} \
  --pred "${OUT_DIR}/lupus_fit_step1_pred.list" \
  --bsize 500 \
  --threads 16 \
  --out "${OUT_DIR}/lupus_all_assoc_"



In [ ]:
%%writefile regenie_test.sh

#!/usr/bin/env bash
set -euo pipefail

# Use REGENIE from your conda env (no need to activate)
export PATH="$HOME/.conda/envs/regenie4/bin:$PATH"


# Show which regenie we will use (sanity check)
echo "Using regenie at: $(which regenie)"


# Local paths

PLINK1="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_step1_regenie"     # prefix only (has .bed/.bim/.fam)

PLINK2="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_v7_plink_files/acaf_threshold.chr16"       # prefix only (has .bed/.bim/.fam)


MERGED_TSV='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt'        # columns: FID IID meningitis sex age

MERGED_TSV2='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt'        # columns: FID IID meningitis sex age

OUT_DIR='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results'

#mkdir -p "$OUT_DIR"


# STEP 2 — single-variant aasoc. test
regenie \
  --step 2 \
  --bed "${PLINK2}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV2}" \
  --phenoCol case \
  --covarFile  "${MERGED_TSV2}" \
  --covarColList sex,age,age2,age_sex,age2_sex,PC{1:16} \
  --pred "${OUT_DIR}/lupus_all_fit_step1_pred.list" \
  --bsize 500 \
  --threads 16 \
  --out "${OUT_DIR}/lupus_all_assoc"



In [ ]:
%%bash

PLINK1="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_step1_regenie"
PLINK2="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_v7_plink_files/acaf_threshold.chr16"

# 1) Backup the current chr16 fam
cp "${PLINK2}.fam" "${PLINK2}.fam.backup"

# 2) Overwrite chr16 fam with the step1 fam (same order, new IDs)
cp "${PLINK1}.fam" "${PLINK2}.fam"

In [ ]:
import pandas as pd

# >>> EDIT THESE TO YOUR ACTUAL FILES <<<
step1_fam = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_step1_regenie.fam"
step2_fam = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_v7_plink_files/acaf_threshold.chr16.fam"
pheno_file = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt"

# --- load FID/IID from fams ---
fam1 = pd.read_csv(step1_fam, delim_whitespace=True, header=None, usecols=[0,1], names=["FID","IID"])
fam2 = pd.read_csv(step2_fam, delim_whitespace=True, header=None, usecols=[0,1], names=["FID","IID"])

# --- load FID/IID from pheno file ---
# assumes first two columns are FID, IID (regenie style)
pheno = pd.read_csv(pheno_file, delim_whitespace=True)

# try to interpret first two columns as FID/IID
pheno_ids = pheno.iloc[:, :2].copy()
pheno_ids.columns = ["FID", "IID"]

set1 = set(map(tuple, fam1[["FID","IID"]].to_numpy()))
set2 = set(map(tuple, fam2[["FID","IID"]].to_numpy()))
setp = set(map(tuple, pheno_ids[["FID","IID"]].to_numpy()))

print("Counts:")
print(f"  Step1 fam IDs : {len(set1)}")
print(f"  Step2 fam IDs : {len(set2)}")
print(f"  Pheno IDs     : {len(setp)}")
print()

# where do pheno IDs land?
p_in_step1 = setp & set1
p_in_step2 = setp & set2
p_in_both  = setp & set1 & set2

print("Pheno ID overlap:")
print(f"  Pheno IDs in step1 fam      : {len(p_in_step1)}")
print(f"  Pheno IDs in step2 fam      : {len(p_in_step2)}")
print(f"  Pheno IDs in BOTH fams      : {len(p_in_both)}")
print(f"  Pheno IDs NOT in step1 fam  : {len(setp - set1)}")
print(f"  Pheno IDs NOT in step2 fam  : {len(setp - set2)}")
print()

# show a few examples if anything is missing
missing_step1 = list(setp - set1)[:5]
missing_step2 = list(setp - set2)[:5]

if missing_step1:
    print("Example pheno IDs NOT in step1 fam (up to 5):")
    for fid, iid in missing_step1:
        print("  ", fid, iid)
    print()

if missing_step2:
    print("Example pheno IDs NOT in step2 fam (up to 5):")
    for fid, iid in missing_step2:
        print("  ", fid, iid)
    print()

# optional: what regenie will effectively use (intersection of all three)
effective_ids = set1 & set2 & setp
print(f"Effective IDs present in step1 fam, step2 fam, AND pheno: {len(effective_ids)}")


In [ ]:
%%bash

chmod +x regenie_test.sh
./regenie_test.sh



In [ ]:
!cat /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results/lupus_all_assoc_case.regenie | head -n20



In [ ]:
%%bash

awk -F'\t' '
NR==1{
  for(i=1;i<=NF;i++) if($i=="case") {c=i; break}
  if(!c){ print "ERROR: header has no \"case\" column"; exit 0}
}
NR>1{
  v=$c
  if(v!="" && v!="NA" && v!="na" && v!="." && v!="-9") nonmiss++
}
END{ print "non-missing case values:", nonmiss+0 }' \
/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL_final.regenie.txt

In [ ]:
%%bash

# create fam id list
awk '{print $1":"$2}' /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_step1_regenie.fam \
  | sort > /tmp/fam_ids_sorted.txt

# pheno: assume first two columns are FID and IID (tab-separated)
awk -F'\t' 'NR>1{print $1":"$2}' /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.mapped.txt \
  | sort > /tmp/pheno_ids_sorted.txt

echo "fam count:" $(wc -l < /tmp/fam_ids_sorted.txt)
echo "pheno count:" $(wc -l < /tmp/pheno_ids_sorted.txt)
echo "overlap count:" $(comm -12 /tmp/fam_ids_sorted.txt /tmp/pheno_ids_sorted.txt | wc -l)

In [ ]:
pheno

In [ ]:


# ---- 1. read fam ----
fam = pd.read_csv(
    f"{results}/acaf_step1_regenie.fam",
    sep=r"\s+",
    header=None,
    names=["FID_fam", "IID_fam", "pat", "mat", "sex_fam", "pheno_fam"]
)

# key = numeric part before underscore in IID_fam
fam["key"] = fam["IID_fam"].astype(str).str.split("_").str[0]

# ---- 2. read phenotype ----
pheno = pd.read_csv(f"{results}/lupus_matched_ALL_final.regenie.txt", sep="\t")

# assume pheno has columns: FID, IID, case, sex, age, ...
# key = IID column as string
pheno["key"] = pheno["IID"].astype(str)

# ---- 3. inner join on key (keeps only samples present in both) ----
merged = pheno.merge(
    fam[["key", "FID_fam", "IID_fam"]],
    on="key",
    how="inner",
)

print("Original pheno rows:", len(pheno))
print("Rows after merging with fam:", len(merged))
print("Dropped rows (pheno not in fam):", len(pheno) - len(merged))

# ---- 4. build final pheno: use fam IDs + original phenotype/covariates ----
# keep all original pheno columns except original FID/IID, and prepend the fam FID/IID
pheno_cols = [c for c in pheno.columns if c not in ["FID", "IID", "key"]]
final_cols = ["FID_fam", "IID_fam"] + pheno_cols

final = merged[final_cols].copy()
final = final.rename(columns={"FID_fam": "FID", "IID_fam": "IID"})

print("\nFinal columns:", list(final.columns))

# ---- 5. write out ----
final.to_csv(f"{results}/lupus_matched_ALL.regenie.mapped.txt", sep="\t", index=False)
print("\nWrote mapped phenotype to:")
print(f"{results}/lupus_matched_ALL.regenie.mapped.txt")

# show a few rows
final.head()

In [ ]:
def map_pheno_to_fam_file_step2(fam_file, pheno_file, final_file):

    # ---- 1. read fam ----
    fam = pd.read_csv(
        fam_file,
        sep=r"\s+",
        header=None,
        names=["FID_fam", "IID_fam", "pat", "mat", "sex_fam", "pheno_fam"]
    )

    # key = numeric part before underscore in IID_fam
    fam["key"] = fam["IID_fam"].astype(str).str.split("_").str[0]

    # ---- 2. read phenotype ----
    pheno = pd.read_csv(pheno_file, sep="\t")

    # assume pheno has columns: FID, IID, case, sex, age, ...
    # key = IID column as string
    pheno["key"] = pheno["IID"].astype(str)

    # ---- 3. inner join on key (keeps only samples present in both) ----
    merged = pheno.merge(
        fam[["key", "FID_fam", "IID_fam"]],
        on="key",
        how="inner",
    )

    print("Original pheno rows:", len(pheno))
    print("Rows after merging with fam:", len(merged))
    print("Dropped rows (pheno not in fam):", len(pheno) - len(merged))

    # ---- 4. build final pheno: use fam IDs + original phenotype/covariates ----
    # keep all original pheno columns except original FID/IID, and prepend the fam FID/IID
    pheno_cols = [c for c in pheno.columns if c not in ["FID", "IID", "key"]]
    final_cols = ["FID_fam", "IID_fam"] + pheno_cols

    final = merged[final_cols].copy()
    final = final.rename(columns={"FID_fam": "FID", "IID_fam": "IID"})

    print("\nFinal columns:", list(final.columns))

    # ---- 5. write out ----
    final.to_csv(final_file, sep="\t", index=False)
    print("\nWrote mapped phenotype to:")
    print(final_file)

    # show a few rows
    final.head()
    
    return fam, pheno, final

In [ ]:
fam1, pheno1, final1 = map_pheno_to_fam_file_step2(f"{data}/acaf_v7_plink_files/acaf_threshold.chr16.fam", f"{results}/lupus_matched_ALL_final.regenie.txt" ,f"{results}/lupus_matched_ALL.regenie.mapped2.txt")

In [ ]:
fam1
pheno1
final1